# Итоговый проект: LLM-ассистент для анализа и обработки пользовательских запросов


**Выбранная прикладная область:** маршрутизация обращений в техническую поддержку

**Описание задачи:**  

**Входные данные.** Короткий свободный текст обращения клиента: что случилось, что уже
пробовали, насколько срочно. Обращения приходят из разных каналов (почта, чат, форма на
сайте), поэтому единого формата у текста нет - разбирать его вручную дорого, а
маршрутизировать надо по формальным правилам.

**Что должен возвращать ассистент.** Строгий JSON из девяти полей, размеченный по
фиксированным перечислениям:

| Поле | Тип | Допустимые значения | Смысл |
|---|---|---|---|
| `category` | enum | `access`, `billing`, `bug`, `performance`, `security`, `question`, `other` | тип проблемы |
| `priority` | enum | `low`, `medium`, `high`, `critical` | в какую очередь ставить |
| `urgency_score` | integer | 1–10 | срочность внутри очереди, для сортировки |
| `department` | enum | `support`, `billing`, `devops`, `security`, `product` | кому отдать обращение |
| `affected_service` | enum | `api`, `web`, `mobile`, `auth`, `payments`, `email`, `other` | что затронуто |
| `is_incident` | boolean | `true` / `false` | сбой на нашей стороне, а не частный вопрос |
| `requires_human` | boolean | `true` / `false` | можно ли ответить автоматически |
| `sentiment` | enum | `negative`, `neutral`, `positive` | тон автора обращения |
| `summary` | string | свободный текст | одна фраза по сути обращения |

**Как используется структурированный результат.** Разметка проставляется тикету при
создании: по `department` обращение уходит нужной команде, по `priority` и `urgency_score`
строится очередь, `is_incident` включает процедуру инцидента, `requires_human` не даёт
автоответу закрыть денежные и security-вопросы.

**Почему поля - перечисления, а не свободный текст.** Значение вне словаря не проходит
валидацию, поэтому фильтры и аналитика дальше пишутся точным сравнением - без `.lower()`
и поиска подстрок. Там, где в тексте данных нет, используется `other`, а не выдуманное
значение.

---

## Чек-лист перед сдачей

- [ ] Все ячейки выполняются без ошибок (Kernel → Restart & Run All)
- [ ] Раздел 2: показана проблема неструктурированного вывода
- [ ] Раздел 3: функция реализована, использует `response_format={'type': 'json_schema', 'schema': …}` (режима `json_object` у GigaChat нет) и `try/except`
- [ ] Раздел 4: пайплайн обрабатывает не менее 5 входных записей
- [ ] Раздел 5: выполнены все 3 задания по фильтрации
- [ ] Раздел 6: сравнение не менее 2 моделей, таблица и письменные выводы
- [ ] Раздел 7: итоги написаны в Markdown
- [ ] API-ключ не хранится в коде - только через `.env`


## 0.1. Получение ключа авторизации GigaChat (Сбер)

Задание выполнено на **GigaChat API** вместо Groq: Groq в РФ недоступен. Различия,
которые учтены дальше по ноутбуку:

* другой SDK - `gigachat` вместо `groq`;
* у GigaChat **нет режима `json_object`**: вместо него передаётся полная JSON-схема
  в `response_format={'type': 'json_schema', 'schema': …, 'strict': True}`;
* API подписан сертификатами НУЦ Минцифры, которых нет в системном хранилище, поэтому
  клиенту нужно явно передать CA-бандл.

**Шаги получения ключа:**

1. Зарегистрировался на **[developers.sber.ru](https://developers.sber.ru)**.
2. Создал проект **GigaChat API** (тариф для физлиц, Free).
3. В настройках проекта скопировал **Client ID** и **Client Secret**.
4. Собрал из них строку авторизации - `Client ID:Client Secret` в Base64:
   `echo -n "CLIENT_ID:CLIENT_SECRET" | base64`.
5. Положил строку в `.env` рядом с ноутбуком:

```
GIGACHAT_CREDENTIALS=СТРОКА_АВТОРИЗАЦИИ
GIGACHAT_CA_BUNDLE_FILE=certs/trusted_bundle.pem
```

**Сертификаты.** Скачиваются и собираются в один бандл скриптом из репозитория:

```bash
bash scripts/fetch_certs.sh
```

> `.env` и `certs/` исключены из git (`.gitignore`) - ключ в репозиторий не попадает.
> Лимиты бесплатного тарифа ограничивают частоту запросов, поэтому в пайплайне между
> вызовами стоит пауза, а на `429` делается повтор.


## 1. Установка и настройка клиента

In [1]:
# Шаг 1.1: Установка библиотек
# В этом репозитории зависимости уже стоят в .venv (см. requirements.txt).
# При запуске в Colab или на чистой машине раскомментируйте строку ниже.
#
# !pip install gigachat python-dotenv


In [2]:
# Шаг 1.2: Импорты и инициализация клиента

import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from gigachat import GigaChat
from gigachat.exceptions import RateLimitError
from gigachat.models import Chat, Messages, MessagesRole

# Ноутбук лежит в корне репозитория, рядом с .env
ENV_PATH = Path(".env")
load_dotenv(ENV_PATH)

CREDENTIALS = os.getenv("GIGACHAT_CREDENTIALS")
CA_BUNDLE = os.getenv("GIGACHAT_CA_BUNDLE_FILE", "certs/trusted_bundle.pem")

if not CREDENTIALS:
    raise RuntimeError(f"В {ENV_PATH.resolve()} нет GIGACHAT_CREDENTIALS — заполните .env по шагу 0.1")
if not Path(CA_BUNDLE).is_file():
    raise RuntimeError(f"Не найден CA-бандл {CA_BUNDLE} — выполните: bash scripts/fetch_certs.sh")

client = GigaChat(
    credentials=CREDENTIALS,
    scope="GIGACHAT_API_PERS",
    ca_bundle_file=CA_BUNDLE,
    timeout=60.0,
)

token = client.get_token()
MODELS_AVAILABLE = [m.id_ for m in client.get_models().data]

# Модель по умолчанию выбираем из фактического списка: имена моделей меняются
DEFAULT_MODEL = next(
    (m for m in ("GigaChat-2", "GigaChat-3-Lightning") if m in MODELS_AVAILABLE),
    MODELS_AVAILABLE[0],
)

# Проверка: выведите первые 5 символов ключа (без полного значения!)
print(f"Ключ загружен: {CREDENTIALS[:5]}… (длина {len(CREDENTIALS)})")
print(f"Токен получен, действует до: {token.expires_at}")
print("Модели на ключе:", ", ".join(MODELS_AVAILABLE))
print("Модель по умолчанию:", DEFAULT_MODEL)


Ключ загружен: MDFhM… (длина 100)
Токен получен, действует до: 1793812772434
Модели на ключе: GigaChat-2, GigaChat-2-Max, GigaChat-2-Pro, GigaChat-3-Lightning, GigaChat-3-Pro, GigaChat-3-Ultra, Embeddings, Embeddings-2, EmbeddingsGigaR, GigaEmbeddings-3B-2025-09
Модель по умолчанию: GigaChat-2


## 2. Проблема: неструктурированный вывод

Попросим модель обработать данные обычным запросом - и посмотрим, почему такой вывод неудобен для программы.

In [3]:
# Шаг 2.1: Задайте один тестовый пример для вашей задачи

sample_input = """
С утра не могу войти в личный кабинет: после ввода пароля страница просто
перезагружается. Браузер Chrome, в режиме инкогнито та же картина.
Доступ нужен срочно — завтра сдавать отчёт. Пароль менял вчера, до этого всё работало.
"""

print(sample_input.strip())


С утра не могу войти в личный кабинет: после ввода пароля страница просто
перезагружается. Браузер Chrome, в режиме инкогнито та же картина.
Доступ нужен срочно — завтра сдавать отчёт. Пароль менял вчера, до этого всё работало.


In [4]:
# Шаг 2.2: Запрос без JSON-режима - покажите, почему это неудобно
#
# Обычный запрос: просим JSON словами, без формального контракта.

response = client.chat(
    Chat(
        model=DEFAULT_MODEL,
        messages=[
            Messages(
                role=MessagesRole.USER,
                content=f"Разбери обращение клиента и верни результат в JSON:\n{sample_input}",
            )
        ],
    )
)

raw_output = response.choices[0].message.content
print("--- сырой ответ модели ---")
print(raw_output)
print()

try:
    data = json.loads(raw_output)
    print("json.loads прошёл, но контракта нет: состав полей и их значения ничем не заданы.")
    print("Ключи, которые выбрала модель:", list(data) if isinstance(data, dict) else type(data).__name__)
except Exception as e:
    print(f"Ошибка json.loads: {e}")
    print("Текст с обрамлением и пояснениями — это не данные, вставить его в код нельзя.")


--- сырой ответ модели ---
```json
{
  "проблема": "Не могу войти в личный кабинет",
  "описание": "После ввода пароля страница перезагружается.",
  "браузер": "Chrome",
  "режим_работы": "Режим инкогнито",
  "состояние_до_проблемы": "Всё работало нормально",
  "действие_перед_проблемой": "Менял пароль вчера",
  "необходимость": "Срочно нужен доступ — завтра сдавать отчёт"
}
```

Ошибка json.loads: Expecting value: line 1 column 1 (char 0)
Текст с обрамлением и пояснениями — это не данные, вставить его в код нельзя.


In [5]:
# Шаг 2.3: Решение - JSON-режим
#
# У Groq это response_format={'type': 'json_object'}: гарантирует «это JSON»,
# но не состав полей. У GigaChat режима json_object нет - вместо него передаётся
# полная схема в response_format={'type': 'json_schema'} + strict: True.
# Слово JSON в системном промпте оставляем (требование шаблона) - оно объясняет
# модели смысл полей, а форму ответа диктует схема.

CATEGORIES = ["access", "billing", "bug", "performance", "security", "question", "other"]
PRIORITIES = ["low", "medium", "high", "critical"]
DEPARTMENTS = ["support", "billing", "devops", "security", "product"]
SERVICES = ["api", "web", "mobile", "auth", "payments", "email", "other"]
SENTIMENTS = ["negative", "neutral", "positive"]

# Схема ответа: контракт между нами и моделью
TICKET_SCHEMA = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": CATEGORIES},
        "priority": {"type": "string", "enum": PRIORITIES},
        "urgency_score": {"type": "integer", "minimum": 1, "maximum": 10},
        "department": {"type": "string", "enum": DEPARTMENTS},
        "affected_service": {"type": "string", "enum": SERVICES},
        "is_incident": {"type": "boolean"},
        "requires_human": {"type": "boolean"},
        "sentiment": {"type": "string", "enum": SENTIMENTS},
        "summary": {"type": "string"},
    },
    "required": [
        "category", "priority", "urgency_score", "department", "affected_service",
        "is_incident", "requires_human", "sentiment", "summary",
    ],
    "additionalProperties": False,
}

SYSTEM_PROMPT = """
Ты — ассистент первой линии технической поддержки. На вход приходит свободный текст
обращения клиента. Разметь обращение и верни строго JSON по схеме.

category — тип проблемы:
  access — вход, пароли, права доступа;
  billing — деньги, счета, подписки, возвраты;
  bug — ошибка или сбой в работе продукта;
  performance — работает, но медленно;
  security — фишинг, утечки, подозрительная активность;
  question — вопрос «как сделать», инструкция, а не поломка;
  other — ничего из перечисленного.

priority — в какую очередь ставить:
  critical — не работает у многих, потеря денег или данных;
  high — блокирует работу клиента;
  medium — мешает, но есть обход;
  low — косметика или пожелание.

urgency_score — целое от 1 до 10, срочность внутри очереди: 1-3 можно подождать,
  4-6 решить в течение дня, 7-8 нужно сегодня, 9-10 немедленно.

department — кому отдать обращение: support, billing, devops, security, product.
affected_service — что затронуто: api, web, mobile, auth, payments, email, other.

is_incident — true только если это сбой или деградация сервиса на нашей стороне.
  Жалоба на списание и вопрос «как выгрузить отчёт» инцидентом не являются.

requires_human — true, если нужен человек: деньги, доступы, безопасность, инциденты.
sentiment — тон автора: negative, neutral, positive.
summary — одна короткая фраза по сути обращения, без домыслов и советов.

Если данных в тексте нет, выбирай other в соответствующем поле. Не придумывай факты.
"""

response = client.chat(
    Chat(
        model=DEFAULT_MODEL,
        messages=[
            Messages(role=MessagesRole.SYSTEM, content=SYSTEM_PROMPT),
            Messages(role=MessagesRole.USER, content=f"Обращение клиента:\n{sample_input}"),
        ],
        response_format={
            "type": "json_schema",
            "schema": TICKET_SCHEMA,
            "strict": True,
        },
        temperature=0.0,
    )
)

raw_json = response.choices[0].message.content
ticket = json.loads(raw_json)

print("--- ответ по схеме ---")
print(json.dumps(ticket, ensure_ascii=False, indent=2))
print()
print(f"полей в ответе: {len(ticket)} из {len(TICKET_SCHEMA['required'])}")
print(f"категория: {ticket['category']} | приоритет: {ticket['priority']} "
      f"| срочность: {ticket['urgency_score']}/10 | отдел: {ticket['department']}")


--- ответ по схеме ---
{
  "category": "access",
  "priority": "high",
  "urgency_score": 8,
  "department": "support",
  "affected_service": "web",
  "is_incident": true,
  "requires_human": true,
  "sentiment": "negative",
  "summary": "Не могу войти в личный кабинет после смены пароля"
}

полей в ответе: 9 из 9
категория: access | приоритет: high | срочность: 8/10 | отдел: support


## 3. Основная функция обработки

Вынесем логику в функцию. Функция должна принимать текст, обращаться к API и возвращать словарь или `None` при ошибке.

In [6]:
# Шаг 3: реализуйте функцию обработки одного элемента
#
# SYSTEM_PROMPT и TICKET_SCHEMA определены выше, в шаге 2.3 - здесь переиспользуем их,
# чтобы определение схемы было в ноутбуке ровно одно.
#
# Требования к функции:
#   - принимает текст (str)
#   - использует системный промпт с описанием схемы JSON
#   - использует response_format={'type': 'json_schema', 'schema': ..., 'strict': True}
#   - обрабатывает ошибки через try/except (минимум: RateLimitError и Exception)
#   - возвращает dict или None

# Системный промпт и схема - см. шаг 2.3 (SYSTEM_PROMPT, TICKET_SCHEMA)


def process_item(text: str, model: str | None = None, attempts: int = 3) -> dict | None:
    """
    Обрабатывает один входной текст с помощью LLM.

    Args:
        text: входной текст для обработки
        model: имя модели; None — модель по умолчанию с ключа
        attempts: сколько раз повторить запрос при лимите запросов

    Returns:
        Словарь с извлечёнными данными или None при ошибке
    """
    chat = Chat(
        model=model or DEFAULT_MODEL,
        messages=[
            Messages(role=MessagesRole.SYSTEM, content=SYSTEM_PROMPT),
            Messages(role=MessagesRole.USER, content=f"Обращение клиента:\n{text}"),
        ],
        response_format={"type": "json_schema", "schema": TICKET_SCHEMA, "strict": True},
        temperature=0.0,
    )

    for attempt in range(1, attempts + 1):
        try:
            response = client.chat(chat)
            return json.loads(response.choices[0].message.content)
        except RateLimitError as e:
            # Лимит частоты на тарифе: ждём столько, сколько просит сервер
            wait = getattr(e, "retry_after", None) or 5
            print(f"Лимит запросов. Ждём {wait} сек... (попытка {attempt}/{attempts})")
            time.sleep(wait)
        except Exception as e:
            print(f"Ошибка: {e}")
            return None

    print(f"Не удалось обработать после {attempts} попыток")
    return None


# Тест на одном примере
test_result = process_item(sample_input)
print(json.dumps(test_result, ensure_ascii=False, indent=2))


{
  "category": "access",
  "priority": "high",
  "urgency_score": 8,
  "department": "support",
  "affected_service": "web",
  "is_incident": true,
  "requires_human": true,
  "sentiment": "negative",
  "summary": "Не могу войти в личный кабинет после смены пароля"
}


## 4. Пайплайн: обработка нескольких элементов

Подготовьте список из **не менее 5** входных записей и прогоните их через функцию.

In [7]:
# Шаг 4.1: подготовьте список входных данных (не менее 5 элементов)
# Семь обращений: разные категории, отделы и степень подробности.
# Одно (T-04) - вопрос, а не поломка; одно (T-07) - «работает, но медленно».

input_items = [
    """
    С утра не могу войти в личный кабинет: после ввода пароля страница просто
    перезагружается. Браузер Chrome, в режиме инкогнито та же картина.
    Доступ нужен срочно — завтра сдавать отчёт. Пароль менял вчера, до этого всё работало.
    """,
    """
    После обновления приложения до версии 2.4.1 раздел «Платежи» открывается
    и сразу закрывается. Так у всей команды на iOS 18. До обновления работало нормально.
    """,
    """
    Списали 1 490 рублей за подписку, хотя я отменил её 12 октября в настройках.
    Прошу вернуть деньги и разобраться, почему списание прошло.
    """,
    """
    Подскажите, пожалуйста, как выгрузить отчёт по продажам за квартал в Excel?
    В интерфейсе искал, но не нашёл.
    """,
    """
    Вчера с 10:00 до 11:20 весь ваш API отдавал 502 на всех запросах. Наши
    интеграции встали, потеряли около 200 заказов. Нужен разбор инцидента.
    """,
    """
    Получил письмо якобы от вашего банка: просят подтвердить карту по ссылке.
    Домен в ссылке странный, не ваш. Похоже на фишинг, хочу предупредить.
    """,
    """
    Дашборд с метриками открывается 8-10 секунд, раньше было около двух.
    Работать можно, но раздражает. Хотелось бы починить.
    """,
]

print(f"записей: {len(input_items)}")


записей: 7


In [8]:
# Шаг 4.2: запустите пайплайн обработки

results = []
failed = []

for number, text in enumerate(input_items, 1):
    print(f"Обрабатываю {number}/{len(input_items)}...")
    parsed = process_item(text)
    if parsed is None:
        failed.append(number)
        continue
    results.append({"id": f"T-{number:02d}", **parsed})
    # пауза между запросами: у бесплатного тарифа ограничена частота
    time.sleep(1.5)

print(f"\nРазмечено: {len(results)} из {len(input_items)}. Сбои: {failed or 'нет'}")

header = f"{'id':<5} {'category':<12} {'priority':<9} {'urg':>3} {'department':<9} {'incident':<9} {'human':<6}"
print()
print(header)
print("-" * len(header))
for r in results:
    print(f"{r['id']:<5} {r['category']:<12} {r['priority']:<9} {r['urgency_score']:>3} "
          f"{r['department']:<9} {str(r['is_incident']):<9} {str(r['requires_human']):<6}")


Обрабатываю 1/7...


Обрабатываю 2/7...


Обрабатываю 3/7...


Обрабатываю 4/7...


Обрабатываю 5/7...


Обрабатываю 6/7...


Обрабатываю 7/7...



Размечено: 7 из 7. Сбои: нет

id    category     priority  urg department incident  human 
------------------------------------------------------------
T-01  access       high        9 support   True      True  
T-02  bug          high        8 devops    True      False 
T-03  billing      high        8 billing   False     True  
T-04  question     medium      3 product   False     False 
T-05  bug          critical   10 devops    True      True  
T-06  security     high        8 security  False     True  
T-07  performance  medium      5 support   False     False 


## 5. Фильтрация и анализ данных

Структурированные данные позволяют фильтровать и анализировать без дополнительных запросов к модели.

Выполните **все 3 задания** ниже, адаптировав их к полям вашей схемы JSON.

In [9]:
# Задание 1: Фильтрация по числовому полю
# Числовое поле схемы - urgency_score (1-10). Порог 7: «нужно сегодня или немедленно».

URGENCY_THRESHOLD = 7

urgent = [r for r in results if r.get("urgency_score", 0) >= URGENCY_THRESHOLD]

print(f"Обращений со срочностью >= {URGENCY_THRESHOLD}: {len(urgent)} из {len(results)}")
for r in sorted(urgent, key=lambda x: x["urgency_score"], reverse=True):
    print(f"  {r['id']} | {r['urgency_score']}/10 | {r['priority']:<8} | {r['department']:<8} | {r['summary']}")


Обращений со срочностью >= 7: 5 из 7
  T-05 | 10/10 | critical | devops   | API выдавал ошибки 502 в течение часа, интеграция потеряла заказы
  T-01 | 9/10 | high     | support  | Не могу войти в личный кабинет после смены пароля
  T-02 | 8/10 | high     | devops   | Ошибка в приложении версии 2.4.1: раздел 'Платежи' закрывается автоматически
  T-03 | 8/10 | high     | billing  | Требуется возврат средств за отменённую подписку
  T-06 | 8/10 | high     | security | Клиент сообщил о подозрительном письме, похожем на фишинг.


In [10]:
# Задание 2: Поиск экстремального значения
# Ищем обращения с максимальной и минимальной срочностью.

if results:
    most_urgent = max(results, key=lambda r: r["urgency_score"])
    least_urgent = min(results, key=lambda r: r["urgency_score"])

    print(f"Самое срочное обращение: {most_urgent['id']} — {most_urgent['urgency_score']}/10")
    print(f"  {most_urgent['category']} / {most_urgent['priority']} / {most_urgent['department']}")
    print(f"  {most_urgent['summary']}")
    print()
    print(f"Самое несрочное обращение: {least_urgent['id']} — {least_urgent['urgency_score']}/10")
    print(f"  {least_urgent['category']} / {least_urgent['priority']} / {least_urgent['department']}")
    print(f"  {least_urgent['summary']}")
else:
    print("Нет размеченных обращений — сначала выполните раздел 4")


Самое срочное обращение: T-05 — 10/10
  bug / critical / devops
  API выдавал ошибки 502 в течение часа, интеграция потеряла заказы

Самое несрочное обращение: T-04 — 3/10
  question / medium / product
  Как выгрузить отчёт по продажам за квартал в Excel?


In [11]:
# Задание 3: Агрегация или статистика
# Сводка по категориям: сколько обращений, средняя и максимальная срочность.

from collections import defaultdict

by_category = defaultdict(list)
for r in results:
    by_category[r["category"]].append(r["urgency_score"])

scores = [r["urgency_score"] for r in results]

if scores:
    print(f"Всего обращений: {len(scores)}")
    print(f"Средняя срочность: {sum(scores) / len(scores):.2f}")
    print(f"min / max: {min(scores)} / {max(scores)}")
    print()
    print(f"{'category':<14} {'обращений':>9} {'средняя urg':>12} {'макс urg':>9}")
    print("-" * 48)
    for category, values in sorted(by_category.items(), key=lambda kv: -len(kv[1])):
        print(f"{category:<14} {len(values):>9} {sum(values) / len(values):>12.2f} {max(values):>9}")
else:
    print("Нет размеченных обращений — сначала выполните раздел 4")


Всего обращений: 7
Средняя срочность: 7.29
min / max: 3 / 10

category       обращений  средняя urg  макс urg
------------------------------------------------
bug                    2         9.00        10
access                 1         9.00         9
billing                1         8.00         8
question               1         3.00         3
security               1         8.00         8
performance            1         5.00         5


## 6. Сравнение моделей

Задание написано под Groq, но Groq в РФ недоступен - работа выполнена на **GigaChat API**.
У Сбера своя линейка: два поколения (2 и 3) и разные классы внутри, поэтому сравнение
«быстрая vs тяжёлая» воспроизводится теми же тремя точками, что и `llama-3.1-8b`
против `llama-3.3-70b` в шаблоне.

Список моделей ниже получен методом `get_models()` для нашего ключа:

| Модель | Поколение | Класс | Ожидаемая роль |
|---|---|---|---|
| `GigaChat-3-Lightning` | 3 | Lightning | минимальная задержка ответа |
| `GigaChat-2` | 2 | базовая | повседневные задачи |
| `GigaChat-2-Pro` | 2 | Pro | ресурсоёмкие задачи |

> Полный список чат-моделей на ключе: `GigaChat-2`, `GigaChat-2-Max`, `GigaChat-2-Pro`,
> `GigaChat-3-Lightning`, `GigaChat-3-Pro`, `GigaChat-3-Ultra`.
> Документация: [developers.sber.ru/docs/ru/gigachat/models](https://developers.sber.ru/docs/ru/gigachat/models)

Сравнение идёт на трёх обращениях (T-01 вход в личный кабинет, T-03 списание денег,
T-05 инцидент с API) - девять запросов, это укладывается в лимиты бесплатного тарифа.
Меряем: время ответа, `usage.total_tokens`, токены в секунду, прошёл ли ответ схему -
и построчно сравниваем поля, чтобы увидеть расхождения в разметке.


In [12]:
# Сравниваем три модели на трёх обращениях.
#
# process_item() возвращает только словарь и скрывает usage, поэтому здесь вызываем
# API напрямую: для замера нужны response.usage и время вокруг вызова.

MODELS_TO_COMPARE = ["GigaChat-2", "GigaChat-2-Pro", "GigaChat-3-Lightning"]
COMPARE_TEXTS = {"T-01": input_items[0], "T-03": input_items[2], "T-05": input_items[4]}

COMPARE_FIELDS = [
    "category", "priority", "urgency_score", "department",
    "affected_service", "is_incident", "requires_human", "sentiment",
]

comparison_results = []

for model in MODELS_TO_COMPARE:
    for ticket_id, text in COMPARE_TEXTS.items():
        chat = Chat(
            model=model,
            messages=[
                Messages(role=MessagesRole.SYSTEM, content=SYSTEM_PROMPT),
                Messages(role=MessagesRole.USER, content=f"Обращение клиента:\n{text}"),
            ],
            response_format={"type": "json_schema", "schema": TICKET_SCHEMA, "strict": True},
            temperature=0.0,
        )
        started = time.perf_counter()
        try:
            response = client.chat(chat)
            elapsed = time.perf_counter() - started
            usage = response.usage
            parsed = json.loads(response.choices[0].message.content)
        except Exception as e:
            elapsed = time.perf_counter() - started
            print(f"{model} / {ticket_id}: Ошибка: {e}")
            comparison_results.append({"model": model, "id": ticket_id,
                                       "seconds": round(elapsed, 2), "total_tokens": 0,
                                       "tok_per_sec": 0.0, "ok": False, "parsed": {}})
            continue

        comparison_results.append({
            "model": model,
            "id": ticket_id,
            "seconds": round(elapsed, 2),
            "total_tokens": usage.total_tokens,
            "tok_per_sec": round(usage.completion_tokens / max(elapsed, 1e-3), 1),
            "ok": True,
            "parsed": parsed,
        })
        # пауза между запросами: ограничена частота на тарифе
        time.sleep(1.5)

header = f"{'model':<21}{'id':<6}{'секунд':>8}{'токенов':>9}{'ток/с':>8}{'json':>6}"
print(header)
print("-" * len(header))
for r in comparison_results:
    print(f"{r['model']:<21}{r['id']:<6}{r['seconds']:>8}{r['total_tokens']:>9}"
          f"{r['tok_per_sec']:>8}{str(r['ok']):>6}")

print()
print("Средние по моделям:")
for model in MODELS_TO_COMPARE:
    rows = [r for r in comparison_results if r["model"] == model and r["ok"]]
    if rows:
        avg_sec = sum(r["seconds"] for r in rows) / len(rows)
        avg_tok = sum(r["total_tokens"] for r in rows) / len(rows)
        print(f"  {model:<21} {avg_sec:>6.2f} с   {avg_tok:>6.0f} токенов")

print()
print("Расхождения в разметке между моделями:")
any_diff = False
for ticket_id in COMPARE_TEXTS:
    diffs = []
    for field in COMPARE_FIELDS:
        values = {r["model"]: r["parsed"].get(field)
                  for r in comparison_results if r["id"] == ticket_id and r["ok"]}
        if len({str(v) for v in values.values()}) > 1:
            diffs.append((field, values))
    if diffs:
        any_diff = True
        print(f"  {ticket_id}:")
        for field, values in diffs:
            joined = " | ".join(f"{m}={v}" for m, v in values.items())
            print(f"    {field}: {joined}")
if not any_diff:
    print("  расхождений нет — все модели разметили одинаково")


model                id      секунд  токенов   ток/с  json
----------------------------------------------------------
GigaChat-2           T-01      0.71      181   127.7  True
GigaChat-2           T-03      0.75      164   122.2  True
GigaChat-2           T-05      0.96      181   103.5  True
GigaChat-2-Pro       T-01      2.39      196    44.3  True
GigaChat-2-Pro       T-03      1.78      175    58.0  True
GigaChat-2-Pro       T-05      1.78      188    59.7  True
GigaChat-3-Lightning T-01      0.65      161   168.5  True
GigaChat-3-Lightning T-03      0.65      141   167.0  True
GigaChat-3-Lightning T-05      0.75      162   157.7  True

Средние по моделям:
  GigaChat-2              0.81 с      175 токенов
  GigaChat-2-Pro          1.98 с      186 токенов
  GigaChat-3-Lightning    0.68 с      155 токенов

Расхождения в разметке между моделями:
  T-01:
    urgency_score: GigaChat-2=8 | GigaChat-2-Pro=8 | GigaChat-3-Lightning=9
    is_incident: GigaChat-2=True | GigaChat-2-Pro=False 

### Анализ результатов сравнения

Прогон: 3 модели × 3 обращения (T-01 вход в личный кабинет, T-03 списание денег,
T-05 инцидент с API), 9 запросов, `temperature=0`. Все числа ниже - из сохранённого
вывода ячейки выше.

1. **Самая быстрая модель** по времени ответа: **`GigaChat-3-Lightning`** - 0.68 с
   в среднем (от 0.65 до 0.75 с на трёх обращениях). `GigaChat-2` - 0.81 с,
   `GigaChat-2-Pro` - 1.98 с, то есть почти втрое медленнее флагманской по скорости.
   **Самая быстрая** по токенам в секунду: **`GigaChat-3-Lightning`** - 164 ток/с
   в среднем (максимум 168.5 на T-01); у `GigaChat-2` - 118, у `GigaChat-2-Pro` - 54.

2. **Качество извлечения.** Структура одинаковая у всех трёх моделей: 9 ответов из 9
   прошли схему, все девять полей на месте, лишних полей нет - `strict: True` держится
   на каждой модели. Расхождения оказались **в значениях**, не в форме:

| Обращение | Поле | GigaChat-2 | GigaChat-2-Pro | GigaChat-3-Lightning | В чём вопрос |
|---|---|---|---|---|---|
| T-01 | `is_incident` | true | **false** | true | сломанный вход в ЛК - инцидент на нашей стороне или частный случай |
| T-01 | `urgency_score` | 8 | 8 | 9 | один балл из десяти |
| T-03 | `department` | billing | billing | **support** | возврат денег должен уходить в `billing` |
| T-05 | `urgency_score` | 10 | 10 | 9 | один балл из десяти, некритично |

**Что из этого следует.** Скорость и качество не идут вместе: самая быстрая модель
(`GigaChat-3-Lightning`, 0.68 с и 164 ток/с) единственная, кто ошибся на деньгах -
увела возврат средств в `support` вместо `billing`. `GigaChat-2` разметила все три
обращения так же, как более тяжёлая `GigaChat-2-Pro`, но втрое быстрее. Выигрыша
в качестве у `GigaChat-2-Pro` на этой задаче не видно, а по скорости она худшая.
Похоже, что для разметки тикетов с жёсткой схемой достаточно базовой модели,
а «тяжёлую» имеет смысл звать только на спорные тексты.

**Про токены.** Средний `usage.total_tokens` на запрос: 175 (`GigaChat-2`), 186
(`GigaChat-2-Pro`), 155 (`GigaChat-3-Lightning`) - величины одного порядка, потому что
в это число входит и промпт со схемой (~150 токенов), и сам ответ. То есть метрика
`total_tokens` здесь измеряет в основном размер промпта, а не «разговорчивость» модели;
для оценки стоимости нужен отдельный подсчёт `completion_tokens`.

> Числа относятся к прогону, сохранённому в этом ноутбуке. При повторном запуске
> значения сдвигаются на единицы процентов, а разметка может отличаться на один класс
> (`temperature=0` не гарантирует полный детерминизм у GigaChat): в разведочном прогоне
> `urgency_score` для T-05 приходил как 10 и 9 у одной и той же модели.


## 7. Итоги и выводы

**Что реализовано.**

* клиент GigaChat API: авторизация по строке `Client ID:Client Secret`, закодированной в Base64,
  корневой сертификат (CA-бандл) НУЦ Минцифры, ключ хранится только в `.env` - в коде его нет;
* демонстрация проблемы: запрос «верни JSON» без контракта возвращает не данные, а текст -
  модель оборачивает ответ в блок кода с меткой `json` и придумывает собственные ключи
  (`проблема`, `описание`, `браузер`, `режим_работы`), поэтому `json.loads` падает;
* решение: `response_format={'type': 'json_schema', 'schema': TICKET_SCHEMA, 'strict': True}`.
  В GigaChat нет режима `json_object`, который предлагает шаблон, поэтому контракт задаётся
  полной JSON-схемой с перечислениями;
* функция `process_item()`: системный промпт, разбор JSON, `try/except`. При `RateLimitError`
  запрос повторяется (до трёх попыток) после паузы, указанной сервером; любая другая ошибка
  возвращает `None`, чтобы одно сбойное обращение не прерывало обработку всего пакета;
* пайплайн на 7 обращениях: размечены все 7, сбоев нет;
* три задания аналитики без обращения к модели: фильтр по `urgency_score >= 7` (5 из 7),
  поиск максимума и минимума (T-05 = 10, T-04 = 3), агрегация по категориям
  (средняя срочность 7.29; `bug` - 2 обращения со средней срочностью 9.00);
* сравнение трёх моделей: 9 запросов, таблица со временем ответа, числом токенов и скоростью
  (ток/с), сравнение значений полей по каждому обращению, текстовые выводы.

**Задача и схема JSON.**

Выбранная область - маршрутизация обращений в техподдержку: на входе свободный текст
обращения, на выходе 9 полей - `category`, `priority`, `urgency_score` (целое 1–10 для
сортировки внутри очереди), `department`, `affected_service`, `is_incident`,
`requires_human`, `sentiment`, `summary`. Классы заданы перечислениями (enum) прямо
в схеме, поэтому значение вне словаря не проходит валидацию, а отсутствие данных
передаётся через `other`, а не через выдуманное значение.

**Что показал прогон.**

* `strict: True` работает на всех трёх моделях: все 9 ответов прошли валидацию по схеме;
* `GigaChat-3-Lightning` быстрее всех (0.68 с, 164 ток/с), но единственная из трёх
  ошиблась в маршрутизации денежного обращения (`support` вместо `billing`);
* `GigaChat-2-Pro` в 2.4 раза медленнее базовой `GigaChat-2` и не даёт выигрыша
  в качестве разметки;
* расхождений между моделями немного, но они содержательные: `is_incident` на пограничном
  случае, `urgency_score` на ±1 балл, один неверный `department`;
* `total_tokens` в основном отражает размер промпта со схемой, а не длину ответа.

**С какими трудностями столкнулись.**

* задание написано под Groq, но в России он недоступен - пришлось перейти на GigaChat
  и разбираться с его авторизацией;
* в GigaChat нет режима `json_object`: вместо одного флага нужна полная JSON-схема
  со `strict: True`;
* `temperature=0` не гарантирует полного детерминизма: при повторных прогонах
  `urgency_score` для T-05 приходил то 10, то 9.

**Возможные улучшения.**

* добавить в схему поле `confidence` и отправлять на ручную проверку обращения с низкой
  уверенностью или с `category: other`;
* закрепить спорные правила прямо в промпте (например: «возврат средств - всегда
  `billing`») и проверить их на размеченной выборке;
* кэшировать ответы по хэшу текста, чтобы повторные прогоны не тратили квоту;
* вынести промпт и схему в отдельный файл: смена предметной области должна требовать
  правки одного файла, а не всего ноутбука;
* в продакшене - ретраи с экспоненциальной паузой, учёт остатка квоты по заголовкам
  ответа и логирование всех `None`, чтобы видеть долю неразмеченных обращений.
